# 49b · check: the disease comparisons of notebook 41 with RUVg factors added

Reads the raw counts and sample table of notebook 40, the negative control genes of notebook 48, and the full
tables `de_<pair>.csv` of notebook 41. Writes `ruv_check_disease_comparisons.csv` in
`data/run_artifacts/CANDLE_SAVI/`.

**Objective.** Notebook 41 compares CANDLE, SAVI, AGS and UAID with each other, with the known batches (the four
facilities, the NIAMS cohorts) as terms in the model. This notebook checks whether adding RUVg factors of unwanted
variation, estimated from the same-lab negative control genes of notebook 48 (Risso D, Ngai J, Speed TP, Dudoit S.
*Nat Biotechnol* 2014;32:896-902), changes those results.

**Reading the check.** If the top genes and log2 fold changes stay the same, the known batch terms already account
for the unwanted variation within these data. If they change, the table shows which genes depend on the choice.
The negative control genes come from SLE studies; among CANDLE, SAVI, AGS and UAID they are assumed, not shown, to
be unchanged.

## The data of notebook 41, and the RUVg factors

In [1]:
source("../src/paths.R")
suppressMessages({library(limma); library(edgeR); library(RUVSeq)})
x <- readRDS(art("CANDLE_SAVI", "counts_raw.rds")); s <- readRDS(art("CANDLE_SAVI", "samples.rds"))
s$diagnosis <- factor(make.names(s$diagnosis)); s$batch <- factor(make.names(s$batch))
design0 <- model.matrix(~ 0 + diagnosis + batch, s); colnames(design0) <- sub("^diagnosis", "", colnames(design0))
dge  <- DGEList(x$counts, genes = x$genes); keep <- filterByExpr(dge, design0)
dge  <- calcNormFactors(dge[keep, , keep.lib.sizes = FALSE])
neg  <- intersect(read.csv(art("CANDLE_SAVI", "control_genes_negative.csv"))$gene_id, rownames(dge))
uq   <- betweenLaneNormalization(dge$counts, which = "upper")
W    <- lapply(setNames(1:3, paste0("k", 1:3)), function(k) RUVg(uq, neg, k = k)$W)
c(samples = ncol(dge), genes = nrow(dge), negative_controls_used = length(neg))

samples                  genes negative_controls_used 
                   160                  28821                    777

**Explanation**
- The counts, sample table, design (`~ 0 + diagnosis + batch`), filter and TMM normalization of notebook 41.
- `neg` holds the negative control genes of notebook 48 among the genes kept here.
- `betweenLaneNormalization()` and `RUVg()` as in notebook 49: `W` keeps the factors of unwanted variation for
  k = 1, 2, 3, estimated from the control genes of these samples only.

**Result.** 160 samples, 28,821 genes; all 777 negative control genes are among them.

## The six comparisons with the factors added

In [2]:
pairs <- c("CANDLE-AGS", "SAVI-AGS", "UAID-AGS", "SAVI-CANDLE", "UAID-CANDLE", "UAID-SAVI")
fit_with <- function(w) {
  d <- cbind(design0, w); colnames(d)[(ncol(design0) + 1):ncol(d)] <- paste0("W", seq_len(ncol(w)))
  v <- voom(dge, d); cr <- duplicateCorrelation(v, d, block = s$patient)$consensus
  fit <- lmFit(v, d, block = s$patient, correlation = cr)
  eBayes(contrasts.fit(fit, makeContrasts(contrasts = pairs, levels = d)))
}
check <- do.call(rbind, lapply(names(W), function(k) { f <- fit_with(W[[k]])
  do.call(rbind, lapply(pairs, function(p) {
    a <- read.csv(art("CANDLE_SAVI", sprintf("de_%s.csv", p)))
    b <- topTable(f, coef = p, number = Inf, sort.by = "none"); b$gene_id <- rownames(b)
    top_a <- a$gene_id[a$adj.P.Val < 0.05 & abs(a$logFC) >= 1]; top_b <- b$gene_id[b$adj.P.Val < 0.05 & abs(b$logFC) >= 1]
    g <- intersect(a$gene_id, b$gene_id)
    data.frame(k = k, comparison = p, top_genes_41 = length(top_a), top_genes_with_W = length(top_b),
               shared = length(intersect(top_a, top_b)), only_41 = length(setdiff(top_a, top_b)), only_with_W = length(setdiff(top_b, top_a)),
               log2FC_pearson = round(cor(a$logFC[match(g, a$gene_id)], b$logFC[match(g, b$gene_id)]), 3))
  }))
}))
write.csv(check, art("CANDLE_SAVI", "ruv_check_disease_comparisons.csv"), row.names = FALSE)
check

k,comparison,top_genes_41,top_genes_with_W,shared,only_41,only_with_W,log2FC_pearson
<chr>,<chr>,<int>,<int>,<int>,<int>,<int>,<dbl>
k1,CANDLE-AGS,1041,938,781,260,157,0.971
k1,SAVI-AGS,538,461,405,133,56,0.995
k1,UAID-AGS,276,43,36,240,7,0.933
k1,SAVI-CANDLE,116,56,54,62,2,0.971
k1,UAID-CANDLE,35,12,10,25,2,0.984
k1,UAID-SAVI,10,1,1,9,0,0.932
k2,CANDLE-AGS,1041,76,66,975,10,0.686
k2,SAVI-AGS,538,1,1,537,0,0.768
k2,UAID-AGS,276,0,0,276,0,0.744


**Explanation**
- `fit_with(w)` fits the model of notebook 41 with the factors `w` as extra columns of the design (`W1`, `W2`,
  ...), with the patient block, and the six comparisons.
- For each k and comparison, the table gives the top genes of notebook 41 and with the factors (adjusted
  p < 0.05, fold change at least 2), how many are shared, how many appear only in one, and the Pearson
  correlation of the log2 fold changes over all genes.
- The table is written as a CSV file.

**Result.**
- **k = 1.** The log2 fold changes stay close to notebook 41 (Pearson 0.93 to 0.995), but fewer genes reach the
  top-gene rule: CANDLE against AGS 938 of 1,041 (781 shared), SAVI against AGS 461 of 538, and UAID against AGS 43
  of 276.
- **k = 2 and 3.** The comparisons with AGS collapse: SAVI against AGS keeps 1 of 538 top genes and UAID against AGS
  none of 276; CANDLE against AGS keeps 76 (k = 2) and 135 (k = 3) of 1,041. SAVI against CANDLE keeps its size
  (116 and 132 top genes, 79 and 83 shared with notebook 41).
- The factors estimated from the SLE-derived control genes follow the differences between the diagnoses, most of
  all between AGS and the others, so adding them to the model removes disease differences. Within these data the
  known batch terms of notebook 41 are the correction to keep; the RUVg factors are not used.

## References

1. Risso D, Ngai J, Speed TP, Dudoit S. Normalization of RNA-seq data using factor analysis of control genes or samples. *Nat Biotechnol* 2014;32:896-902.
2. Gagnon-Bartsch JA, Speed TP. Using control genes to correct for unwanted variation in microarray data. *Biostatistics* 2012;13:539-552.
3. Law CW, Chen Y, Shi W, Smyth GK. voom: precision weights unlock linear model analysis tools for RNA-seq read counts. *Genome Biol* 2014;15:R29.
4. Smyth GK, Michaud J, Scott HS. Use of within-array replicate spots for assessing differential expression in microarray experiments. *Bioinformatics* 2005;21:2067-2075.
5. Ritchie ME, Phipson B, Wu D, Hu Y, Law CW, Shi W, Smyth GK. limma powers differential expression analyses for RNA-sequencing and microarray studies. *Nucleic Acids Res* 2015;43:e47.